# Comparison Model: RF, GAM, and GAQQ (Mutation Level)

## What this notebook does:
This notebook evaluates and compares the performance of three predictive models (Random Forest, GAM, and GAQQ via R) in forecasting antibiotic resistance. It applies these models to mutation-level data, segmenting the analysis by essential genes, nonessential genes, and a combined dataset. Finally, it evaluates the forecasting performance on Category-3 (uncertain significance) variants from 2021 that were reclassified in the 2023 catalog.

## Main Input:
- `./2021/2021_final_df.csv`: The primary training and evaluation dataset from the 2021 catalog.
- `./2023/2023_final_df.csv`: The updated 2023 catalog dataset, used strictly for finding the true labels of reclassified Category-3 variants.
- `./Comparison_Model/est.twoclass.R`: The R script defining the SLDA.EBIC and classification logic for the GAQQ model.

## Main Output:
- Trained model objects for RF, GAM, and GAQQ.
- A comprehensive summary DataFrame (`out_df`) detailing the ROC AUC, Sensitivity, Specificity, F1-scores, and Youden-J optimal thresholds for each model and dataset.

## How this notebook fits into the workflow
Run this notebook after `forecast_data_preparation_combined.ipynb`. It depends on the derived feature tables produced there.

In [3]:
import os
import numpy as np
import pandas as pd
import warnings
import math

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_curve, roc_auc_score, confusion_matrix, classification_report
)
from sklearn.ensemble import RandomForestClassifier

# --- GAM ---
from pygam import LogisticGAM

# --- GAQQ via R (rpy2) ---
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

warnings.filterwarnings('ignore')
pandas2ri.activate()
r = ro.r

# Define paths matching the GitHub repository structure
EST_TWOCLASS_R = "./Comparison_Model/est.twoclass.R"

# Load the R script
try:
    r['source'](EST_TWOCLASS_R)
    print("R script loaded successfully.")
except Exception as e:
    print(f"Error loading R script. Please ensure {EST_TWOCLASS_R} exists. Error: {e}")

R[write to console]: Loading required package: Matrix

R[write to console]: Loaded glmnet 4.1-8



R script loaded successfully.


## Load and quality-control the 2021 training table

The next cells load the 2021 feature table, remove duplicate rows, and restrict the analysis to variants with valid structural proximity values, since `Prox_3D_zeroed` is part of the final 25-feature supervised model matrix.

In [4]:

FILE_2021 = "./2021/2021_final_df.csv"
catalog_data = pd.read_csv(FILE_2021)
print("catalog data shape", catalog_data.shape)
#drop duplicates
catalog_data = catalog_data.drop_duplicates()
print("debug: check features", catalog_data.columns)
catalog_data = catalog_data[catalog_data['Prox_3D'].notna()]
print("debug: catalog data confidence distribution", catalog_data['confidence'].value_counts())
catalog_data.isna().sum()

catalog data shape (4709, 50)
debug: check features Index(['gene', 'drug', 'confidence', 'phenotype', 'mutation_oneletter',
       'mutation_wt', 'mutation_pos', 'mutation_mut', 'freq_variant',
       'Rosetta_fa_atr', 'Rosetta_fa_rep', 'Rosetta_fa_sol', 'Rosetta_fa_elec',
       'Rosetta_fa_dun', 'Rosetta_ddG', 'DeltaZ', 'Prox_WHO_Adjusted_Pos',
       'Prox_1D', 'Prox_1D_nearest', 'Prox_3D', 'Prox_3D_nearest',
       'Prox_3D_zeroed', 'LLR_score', 'LLR_dim33', 'LLR_dim46', 'LLR_dim62',
       'LLR_dim70', 'LLR_dim124', 'LLR_dim192', 'LLR_dim207', 'LLR_dim258',
       'LLR_dim267', 'LLR_dim315', 'AAIndex_delta1', 'AAIndex_delta2',
       'AAIndex_delta3', 'AAIndex_delta4', 'AAIndex_delta5', 'AAIndex_delta6',
       'AAIndex_delta7', 'AAIndex_delta8', 'AAIndex_mut1', 'AAIndex_mut2',
       'AAIndex_mut3', 'AAIndex_mut4', 'AAIndex_mut5', 'AAIndex_mut6',
       'AAIndex_mut7', 'AAIndex_mut8', 'mutation'],
      dtype='object')
debug: catalog data confidence distribution 3) Uncertain sign

gene                        0
drug                        0
confidence                  0
phenotype                   0
mutation_oneletter          0
mutation_wt                 0
mutation_pos                0
mutation_mut                0
freq_variant                5
Rosetta_fa_atr              0
Rosetta_fa_rep              0
Rosetta_fa_sol              0
Rosetta_fa_elec             0
Rosetta_fa_dun              0
Rosetta_ddG                 0
DeltaZ                      0
Prox_WHO_Adjusted_Pos    3631
Prox_1D                     0
Prox_1D_nearest             0
Prox_3D                     0
Prox_3D_nearest             0
Prox_3D_zeroed              0
LLR_score                   0
LLR_dim33                   0
LLR_dim46                   0
LLR_dim62                   0
LLR_dim70                   0
LLR_dim124                  0
LLR_dim192                  0
LLR_dim207                  0
LLR_dim258                  0
LLR_dim267                  0
LLR_dim315                  0
AAIndex_de

## prepare training data

In [5]:


def create_label_data(catalog_data):
    """
    Prepare labeled data for essential, nonessential, and combined models.
    Prints which genes are *fully uncertain* (excluded) vs have labeled entries.
    """

    # ─── Identify per-gene confidence composition ─────────────────────
    conf_by_gene = catalog_data.groupby("gene")["confidence"].unique().to_dict()

    fully_uncertain = [
        g for g, confs in conf_by_gene.items()
        if set(confs) == {"3) Uncertain significance"}
    ]
    partially_or_fully_labeled = [
        g for g, confs in conf_by_gene.items()
        if "3) Uncertain significance" not in set(confs) or
           len(set(confs) - {"3) Uncertain significance"}) > 0
    ]

    print(" Genes with *only* uncertain confidence (excluded from training):")
    print(fully_uncertain if fully_uncertain else "None found.")

    print("\n Genes with labeled (certain) entries (used for training):")
    print(partially_or_fully_labeled if partially_or_fully_labeled else "None found.")

    # ─── Keep only non-uncertain rows for training ────────────────────
    label_data = catalog_data[catalog_data["confidence"] != "3) Uncertain significance"].copy()
    print(f"\nFiltered to non-uncertain entries: {label_data.shape[0]} rows")

    # ─── Binary label mapping ─────────────────────────────────────────
    label_data["binary_confidence"] = label_data["confidence"].map({
        "1) Assoc w R": 1,
        "2) Assoc w R - Interim": 1,
        "4) Not assoc w R - Interim": 0,
        "5) Not assoc w R": 0,
    })
    label_data = label_data[label_data["binary_confidence"].notna()].copy()

    # ─── Collapse repeated same-phenotype cross-drug entries ─────────
    pre_collapse_n = label_data.shape[0]
    dup_counts = (
        label_data.groupby(["gene", "mutation_oneletter", "binary_confidence"])
        .size()
        .reset_index(name="n_rows")
    )
    repeated_groups = dup_counts[dup_counts["n_rows"] > 1].copy()
    print(f"\nRepeated gene+mutation+label groups before collapse: {repeated_groups.shape[0]}")
    if not repeated_groups.empty:
        print(repeated_groups.sort_values(["gene", "mutation_oneletter"]).head(10).to_string(index=False))

    label_data = label_data.drop_duplicates(
        subset=["gene", "mutation_oneletter", "binary_confidence"]
    ).copy()
    print(f"Collapsed labeled supervised set: {pre_collapse_n} -> {label_data.shape[0]} rows")

    # ─── Drop unused or redundant columns ─────────────────────────────
    drop_cols = [
        "mutation_wt", "mutation_pos", "mutation_mut",
        "drug", "mutation_oneletter",
        "DeltaZ", "freq_variant",
        "Prox_WHO_Adjusted_Pos",   # may have NAs; that’s fine
        "Prox_1D","Prox_1D_nearest",
        "Prox_3D", "Prox_3D_nearest",  # keep Prox_3D_zeroed
        "LLR_score","mutation"
    ]
    drop_cols += [c for c in label_data.columns if c.startswith("AAIndex_delta")]

    # ─── Identify numeric feature columns ─────────────────────────────
    numeric_columns = [
        c for c in label_data.columns
        if c not in drop_cols and c not in ["confidence", "phenotype", "gene", "binary_confidence", "gene_norm", "essentiality"]
    ]
    print(f"\nNumeric feature count (after dropping metadata): {len(numeric_columns)}")

    # ─── NA inspection (strict) ───────────────────────────────────────
    na_cols = label_data[numeric_columns].columns[label_data[numeric_columns].isna().any()]
    if len(na_cols) > 0:
        print("\n Columns with missing values:")
        for col in na_cols:
            print(f"  - {col}: {label_data[col].isna().sum()} missing")
        print(" If only 'Prox_WHO_Adjusted_Pos' has NAs, this is expected.")
    else:
        print("\n No missing values detected in numeric columns.")

    print(f"\nFinal training-ready collapsed dataset: {label_data.shape[0]} rows, {len(numeric_columns)} features")

    # ─── Essentiality split ───────────────────────────────────────────
    nonessential_genes = ["pncA", "gid", "ethA"]
    label_data["gene_norm"] = label_data["gene"].str.lower()
    nonessential_norm = [g.lower() for g in nonessential_genes]

    label_data["essentiality"] = np.where(
        label_data["gene_norm"].isin(nonessential_norm),
        "nonessential",
        "essential",
    )

    essential_df = label_data[label_data["essentiality"] == "essential"].copy()
    nonessential_df = label_data[label_data["essentiality"] == "nonessential"].copy()

    X_ess, y_ess = essential_df[numeric_columns], essential_df["binary_confidence"]
    X_noness, y_noness = nonessential_df[numeric_columns], nonessential_df["binary_confidence"]
    X_combined, y_combined = label_data[numeric_columns], label_data["binary_confidence"]

    # ─── Summary ──────────────────────────────────────────────────────
    print("\n Dataset Summary:")
    print(f"  Essential:     {X_ess.shape[0]} samples")
    print(f"  Nonessential:  {X_noness.shape[0]} samples")
    print(f"  Combined:      {X_combined.shape[0]} samples")

    return (
        X_ess, y_ess,
        X_noness, y_noness,
        X_combined, y_combined,
        label_data,            #  include this
        numeric_columns        # optionally keep track of feature list
    )

 
X_ess, y_ess, X_noness, y_noness, X_combined, y_combined, label_data, numeric_columns = create_label_data(catalog_data)

print("Essential:", X_ess.shape, y_ess.value_counts().to_dict())
print("Nonessential:", X_noness.shape, y_noness.value_counts().to_dict())
print("Combined:", X_combined.shape, y_combined.value_counts().to_dict())



 Genes with *only* uncertain confidence (excluded from training):
['ddn', 'rplC']

 Genes with labeled (certain) entries (used for training):
['embB', 'ethA', 'gid', 'gyrA', 'gyrB', 'inhA', 'katG', 'pncA', 'rpoB', 'rpsL', 'tlyA']

Filtered to non-uncertain entries: 382 rows

Repeated gene+mutation+label groups before collapse: 25
gene mutation_oneletter  binary_confidence  n_rows
gyrA              A384V                0.0       2
gyrA              A463S                0.0       2
gyrA               A90G                0.0       2
gyrA               A90V                1.0       2
gyrA               D94A                1.0       2
gyrA               D94G                1.0       2
gyrA               D94H                1.0       2
gyrA               D94N                1.0       2
gyrA               D94Y                1.0       2
gyrA               E21Q                0.0       2
Collapsed labeled supervised set: 370 -> 345 rows

Numeric feature count (after dropping metadata): 25

 No

In [6]:
label_data.shape

(345, 53)

## prepare forecasting evaluation data (reclassified variants)

In [7]:

# def merge_catalogs(df_2021, df_2023):
#     merged_df = pd.merge(
#         df_2021, df_2023,
#         on=["gene", "mutation_oneletter"],
#         suffixes=("_2021", "_2023")
#     )
#     return merged_df.drop_duplicates()


# def reclassified_variants(merged_df, catalog_data, X_ref_cols):
#     confidence_3_changed = merged_df[
#         (merged_df["confidence_2021"] == "3) Uncertain significance") &
#         (merged_df["confidence_2023"] != "3) Uncertain significance")
#     ].drop_duplicates(subset=["gene", "mutation_oneletter", "confidence_2021", "confidence_2023"])

#     catalog_data = catalog_data.drop_duplicates(subset=["gene", "mutation_oneletter", "confidence"])
#     selected_positions = confidence_3_changed[["gene", "mutation_oneletter", "confidence_2023"]]

#     category_3_data = catalog_data[catalog_data["confidence"] == "3) Uncertain significance"].copy()
#     category_3_data = category_3_data.merge(selected_positions, on=["gene", "mutation_oneletter"], how="inner")
    
#     if "Prox_WHO_Adjusted_Pos" in category_3_data.columns:
#         category_3_data = category_3_data.drop("Prox_WHO_Adjusted_Pos", axis=1)

#     na_summary = category_3_data[X_ref_cols].isna().sum()

#     return {
#         "confidence_3_changed": confidence_3_changed,
#         "category_3_data": category_3_data,
#         "na_summary": na_summary
#     }
def collapse_to_mutation_level(df, keep='first'):
    """
    Collapse catalogue rows to one record per gene + mutation_oneletter.
    Used for mutation-level temporal evaluation and mutation-level deployment forecasting.
    """
    out = df.copy()
    out = out.drop_duplicates()
    return out.drop_duplicates(subset=['gene', 'mutation_oneletter'], keep=keep).copy()


def collapse_confidence_to_mutation_level(df, confidence_col):
    """
    Collapse drug-specific WHO confidence rows to one mutation-level confidence call.

    Rule:
    - if any non-Category-3 label exists for a mutation in that year, treat the mutation as labeled
    - choose a representative label by priority within the available labels
    - otherwise keep Category 3
    """
    priority = {
        '1) Assoc w R': 0,
        '2) Assoc w R - Interim': 1,
        '4) Not assoc w R - Interim': 2,
        '5) Not assoc w R': 3,
        '3) Uncertain significance': 4,
    }

    rows = []
    for (gene, mut), sub in df.groupby(['gene', 'mutation_oneletter'], dropna=False):
        confs = [c for c in sub[confidence_col].dropna().tolist()]
        non_uncertain = [c for c in confs if c != '3) Uncertain significance']
        pool = non_uncertain if non_uncertain else confs
        chosen = sorted(pool, key=lambda c: priority.get(c, 999))[0] if pool else np.nan
        first = sub.iloc[0].copy()
        first[confidence_col] = chosen
        rows.append(first)
    return pd.DataFrame(rows).reset_index(drop=True)



def merge_catalogs(df_2021, df_2023):
    """
    Collapse both WHO catalogues to mutation level, then merge on gene + mutation.
    """
    df_2021_mut = collapse_confidence_to_mutation_level(df_2021, 'confidence')
    df_2023_mut = collapse_confidence_to_mutation_level(df_2023, 'confidence')

    merged_df = pd.merge(
        df_2021_mut,
        df_2023_mut,
        on=['gene', 'mutation_oneletter'],
        suffixes=('_2021', '_2023')
    )
    return merged_df.drop_duplicates()



def reclassified_variants(merged_df, catalog_data, X_ess):
    """
    Identify mutation-level variants whose WHO confidence changed between 2021 and 2023,
    focusing on mutations that moved out of Category 3.
    """
    unchanged_cat3 = merged_df[
        (merged_df['confidence_2021'] == '3) Uncertain significance') &
        (merged_df['confidence_2023'] == '3) Uncertain significance')
    ].drop_duplicates()

    filtered_df = merged_df[
        ~((merged_df['confidence_2021'] == '3) Uncertain significance') &
          (merged_df['confidence_2023'] == '3) Uncertain significance'))
    ].drop_duplicates()

    updated_conf = filtered_df[
        filtered_df['confidence_2021'] != filtered_df['confidence_2023']
    ].drop_duplicates()

    confidence_3_changed = filtered_df[
        (filtered_df['confidence_2021'] == '3) Uncertain significance') &
        (filtered_df['confidence_2023'] != '3) Uncertain significance')
    ].drop_duplicates(subset=['gene', 'mutation_oneletter', 'confidence_2021', 'confidence_2023'])

    category_3_data = catalog_data[catalog_data['confidence'] == '3) Uncertain significance'].copy()
    category_3_data = collapse_to_mutation_level(category_3_data)
    selected_positions = confidence_3_changed[['gene', 'mutation_oneletter', 'confidence_2023']].copy()
    category_3_data = category_3_data.merge(
        selected_positions,
        on=['gene', 'mutation_oneletter'],
        how='inner'
    )
    if 'Prox_WHO_Adjusted_Pos' in category_3_data.columns:
        category_3_data = category_3_data.drop('Prox_WHO_Adjusted_Pos', axis=1)

    na_summary = category_3_data[X_ess.columns].isna().sum()

    return {
        'unchanged_cat3': unchanged_cat3,
        'updated_conf': updated_conf,
        'confidence_3_changed': confidence_3_changed,
        'changed_counts': confidence_3_changed['confidence_2023'].value_counts(),
        'category_3_data': category_3_data,
        'na_summary': na_summary,
    }


In [8]:
# Load the 2021 and 2023 datasets

file_2021 = "./2021/2021_final_df.csv"
file_2023 = "./2023/2023_final_df.csv"

df_2021 = pd.read_csv(file_2021)
df_2021 = df_2021[df_2021['Prox_3D'].notna()]
df_2023 = pd.read_csv(file_2023)
df_2023 = df_2023[df_2023['Prox_3D'].notna()]

# Step 1: merge
merged_df = merge_catalogs(df_2021, df_2023)

# Step 2: run reclassification analysis
reclassified_eval = reclassified_variants(merged_df, catalog_data, X_ess)

print("Unchanged Cat 3:", reclassified_eval["unchanged_cat3"].shape[0])
print("Changed Cat 3:", reclassified_eval["confidence_3_changed"].shape[0])
print("Counts of new categories:", reclassified_eval["changed_counts"])
reclassified_eval["category_3_data"]['gene'].value_counts()

Unchanged Cat 3: 3114
Changed Cat 3: 62
Counts of new categories: 2) Assoc w R - Interim        44
1) Assoc w R                  13
4) Not assoc w R - Interim     4
5) Not assoc w R               1
Name: confidence_2023, dtype: int64


pncA    46
gid      5
ethA     4
gyrB     4
katG     2
gyrA     1
Name: gene, dtype: int64

In [9]:
## Supervised model training and holdout evaluation

In [10]:
def youden_threshold(y_true, y_score, default=0.5):
    if len(np.unique(y_true)) < 2:
        return float(default)
    fpr, tpr, thr = roc_curve(y_true, y_score)
    j = tpr - fpr
    cand = float(thr[np.argmax(j)])
    return cand if np.isfinite(cand) else float(default)


def summarize_binary(y_true, y_pred, y_score, label, model_name, results_summary):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    roc_auc = roc_auc_score(y_true, y_score)

    report_dict = classification_report(
        y_true, y_pred, labels=[0, 1],
        target_names=["Susceptible", "Resistant"],
        output_dict=True, zero_division=0
    )
    incorrect_count = int((y_true != y_pred).sum())

    results_summary.append({
        "Feature Set": label,
        "Model": model_name,
        "ROC AUC": 100 * roc_auc,
        "Sensitivity (Res)": 100 * sensitivity,
        "Specificity (Sus)": 100 * specificity,
        "Precision (Res)": 100 * report_dict["Resistant"]["precision"],
        "Recall (Res)": 100 * report_dict["Resistant"]["recall"],
        "F1 (Res)": 100 * report_dict["Resistant"]["f1-score"],
        "Precision (Sus)": 100 * report_dict["Susceptible"]["precision"],
        "Recall (Sus)": 100 * report_dict["Susceptible"]["recall"],
        "F1 (Sus)": 100 * report_dict["Susceptible"]["f1-score"],
        "Support (Res)": int(report_dict["Resistant"]["support"]),
        "Support (Sus)": int(report_dict["Susceptible"]["support"]),
        "Incorrect Predictions": incorrect_count
    })

In [11]:
import math


import math
from math import sqrt

def wilson_interval(successes, total, z=1.96):
    if total == 0:
        return (float('nan'), float('nan'))
    p = successes / total
    denom = 1 + z**2 / total
    center = p + z*z/(2*total)
    margin = z * sqrt((p*(1-p) + z*z/(4*total)) / total)
    lower = (center - margin) / denom
    upper = (center + margin) / denom
    return lower, upper


def predict_and_evaluate(
    model,
    category_3_data,
    X_ref,
    catalog_data,
    label="essential",
    model_name="Logistic Regression",
    results_summary=None,
    save_dir="./results",
    save_files=True,
    fixed_threshold=None,
):

    """
    Evaluate a trained model on Category 3 variants using a *fixed* decision threshold
    (derived from training) to avoid look-ahead bias. Computes AUC/sensitivity/specificity,
    saves outputs, and optionally appends to a summary list.
    """
    os.makedirs(save_dir, exist_ok=True)

    os.makedirs(os.path.join(save_dir, "incorrect_predictions"), exist_ok=True)

    # ─── Step 1: Prepare inputs ─────────────────────────────────────────
    X_category_3 = category_3_data[X_ref.columns].fillna(0)
    if hasattr(model, "predict_proba"):
        y_probs = model.predict_proba(X_category_3)[:, 1]
    else:
        y_scores = model.decision_function(X_category_3)
        rng = y_scores.max() - y_scores.min()
        y_probs = (y_scores - y_scores.min()) / (rng if rng != 0 else 1.0)

    category_mapping = {
        "1) Assoc w R": 1, "2) Assoc w R - Interim": 1,
        "4) Not assoc w R - Interim": 0, "5) Not assoc w R": 0
    }
    y_true = category_3_data["confidence_2023"].map(category_mapping)

    # ─── Step 2: Threshold (fixed from training) ───────────────────────
    if fixed_threshold is None or (isinstance(fixed_threshold, float) and math.isnan(fixed_threshold)):
        raise ValueError("predict_and_evaluate requires a fixed_threshold computed from training; do not derive thresholds from evaluation labels.")

    threshold = float(fixed_threshold)
    print(f"[INFO] Using provided threshold for {label}: {threshold:.3f}")

    y_pred = (y_probs >= threshold).astype(int)
    category_3_data = category_3_data.copy()
    category_3_data["true_label"] = y_true
    category_3_data["model_prediction"] = y_pred
    category_3_data["correct"] = y_pred == y_true

    # ─── Step 3: Compute metrics ───────────────────────────────────────
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0

    unique_labels = np.unique(y_true)
    if len(unique_labels) < 2:
        roc_auc = float('nan')
        print("[WARN] Only one class present in evaluation set; ROC AUC set to NaN.")
    else:
        roc_auc = roc_auc_score(y_true, y_probs)

    accuracy = category_3_data["correct"].mean() * 100

    report_dict = classification_report(
        y_true, y_pred, labels=[0, 1],
        target_names=["Susceptible", "Resistant"],
        output_dict=True, zero_division=0
    )

    incorrect_count = (~category_3_data["correct"]).sum()
    total = len(y_true)
    n_res = int((y_true == 1).sum())
    n_sus = int((y_true == 0).sum())

    
    # Wilson intervals
    sens_low, sens_high = wilson_interval(tp, tp + fn) if (tp + fn) else (float('nan'), float('nan'))
    spec_low, spec_high = wilson_interval(tn, tn + fp) if (tn + fp) else (float('nan'), float('nan'))

    # ─── Step 4: Save files (optional) ──────────────────────────────────
    out_csv = os.path.join(save_dir, f"{label}_{model_name}_predictions_on_category3.csv")
    incorrect_path = os.path.join(save_dir, "incorrect_predictions",
                                  f"{label}_{model_name}_incorrect_predictions.csv")


    if save_files:
        category_3_data.to_csv(out_csv, index=False)
        category_3_data.loc[~category_3_data["correct"]].to_csv(incorrect_path, index=False)


    # ─── Step 5: Append to results summary ──────────────────────────────
    if results_summary is not None:
        results_summary.append({
            "Feature Set": label,
            "Model": model_name,
            "Optimal Threshold": round(threshold, 3),
            "Accuracy (%)": accuracy,
            "ROC AUC": 100 * roc_auc if not math.isnan(roc_auc) else float('nan'),
            "Sensitivity (Res)": 100 * sensitivity,
            "Specificity (Sus)": 100 * specificity,
            "Precision (Res)": 100 * report_dict["Resistant"]["precision"],
            "Recall (Res)": 100 * report_dict["Resistant"]["recall"],
            "F1 (Res)": 100 * report_dict["Resistant"]["f1-score"],
            "Precision (Sus)": 100 * report_dict["Susceptible"]["precision"],
            "Recall (Sus)": 100 * report_dict["Susceptible"]["recall"],
            "F1 (Sus)": 100 * report_dict["Susceptible"]["f1-score"],
            "Support (Res)": int(report_dict["Resistant"]["support"]),
            "Support (Sus)": int(report_dict["Susceptible"]["support"]),
            "Incorrect Predictions": int(incorrect_count),
            "Sensitivity CI (95%)": (round(100*sens_low,2), round(100*sens_high,2)) if not math.isnan(sens_low) else (float("nan"), float("nan")),
            "Specificity CI (95%)": (round(100*spec_low,2), round(100*spec_high,2)) if not math.isnan(spec_low) else (float("nan"), float("nan")),
        })
    plot_feature_distribution(category_3_data, model_name, label)
    plot_confusion_matrix(cm, label, model_name)
    plot_false_positives_by_gene(category_3_data, model_name, label)

    # ─── Step 6: Print quick summary ────────────────────────────────────
    print(f"{label.upper()} MODEL SUMMARY:")
    print(f"  Samples: total={total} (Res={n_res}, Sus={n_sus})")
    print(f"  AUC: {roc_auc:.3f}")
    print(f"  Sensitivity: {sensitivity:.2f} | Specificity: {specificity:.2f} | Accuracy: {accuracy:.1f}%")
    print(f"  95% CI (Sens): {100*sens_low:.1f}–{100*sens_high:.1f}; (Spec): {100*spec_low:.1f}–{100*spec_high:.1f}")
    print(f"  Incorrect Predictions: {incorrect_count}")

    # ─── Step 7: Return structured results ─────────────────────────────
    return {
        "label": label,
        "Optimal Threshold": round(threshold, 3),
        "ROC AUC": 100 * roc_auc if not math.isnan(roc_auc) else float('nan'),
        "Accuracy (%)": accuracy,
        "Sensitivity (Res)": 100 * sensitivity,
        "Specificity (Sus)": 100 * specificity,
        "Precision/Recall/F1": report_dict,
        "Confusion Matrix": {"TP": tp, "FP": fp, "FN": fn, "TN": tn},
        "Incorrect Predictions": int(incorrect_count),
        "Saved Prediction CSV": out_csv if save_files else None,
        "Saved Incorrect CSV": incorrect_path if save_files else None,
        "Sample counts": {"total": total, "resistant": n_res, "susceptible": n_sus},
    }


##  Run comparism model

In [12]:

import numpy as np
import pandas as pd
import math
from math import sqrt
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier


def run_experiment_extended(X, y, model_name="rf", mode="holdout", label="Essential", 
                            test_size=0.2, random_state=42, n_splits=5, n_repeats=3):
    feature_names = X.columns
    y = np.asarray(y).astype(int)
 
    train_idx, test_idx = train_test_split(
        np.arange(len(X)), test_size=test_size, stratify=y, random_state=random_state
    )
    X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
 
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    class_counts = np.bincount(y_train)
    min_class = class_counts[class_counts > 0].min() if len(class_counts) else 0
    inner_splits = max(2, min(n_splits, int(min_class))) if min_class >= 2 else 2

    cv = RepeatedStratifiedKFold(n_splits=inner_splits, n_repeats=n_repeats, random_state=random_state)

     
    def get_cv_threshold(model_type):
        thresholds = []
        for tr_idx, val_idx in cv.split(X_train_raw, y_train): 
            X_tr_raw, X_val_raw = X_train_raw.iloc[tr_idx], X_train_raw.iloc[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            if model_type == "rf": 
                m = RandomForestClassifier(n_estimators=200, random_state=random_state, class_weight="balanced_subsample")
                m.fit(X_tr_raw, y_tr)
                probs = m.predict_proba(X_val_raw)[:, 1]
                
            elif model_type == "gam": 
                local_scaler = StandardScaler()
                X_tr_s = local_scaler.fit_transform(X_tr_raw)
                X_val_s = local_scaler.transform(X_val_raw)
                m = LogisticGAM()
                m.fit(X_tr_s, y_tr)
                probs = m.predict_proba(X_val_s)
                
            elif model_type == "gaqq": 
                local_scaler = StandardScaler()
                X_tr_s = local_scaler.fit_transform(X_tr_raw)
                X_val_s = local_scaler.transform(X_val_raw)

                SLDA_EBIC = ro.globalenv["SLDA.EBIC"]
                classify = ro.globalenv["classify"]
                X1_tr, X2_tr = X_tr_s[y_tr == 1], X_tr_s[y_tr == 0]
                res = SLDA_EBIC(X1_tr, X2_tr, np.linspace(0.01, 5, 50), np.linspace(0.01, 5, 50))
                C_opt = np.array(res.rx2("C_opt"))
                dh_opt = np.array(res.rx2("delta_h_opt"))
                C_reg = ro.r.matrix(C_opt + 1e-8 * np.eye(C_opt.shape[0]), nrow=C_opt.shape[0], ncol=C_opt.shape[1])

                X3_val, X4_val = X_val_s[y_val == 1], X_val_s[y_val == 0]
                pred_res = classify(X1_tr, X2_tr, X3_val, X4_val, C_reg, dh_opt)

                if "y_hat" in pred_res.names:
                    y_hat = np.array(pred_res.rx2("y_hat")).astype(float)
                    probs = (y_hat - y_hat.min()) / (y_hat.max() - y_hat.min() + 1e-12)
                else:
                    probs = 1.0 - np.array(pred_res.rx2("est_label")).astype(float)
                y_val = np.concatenate([np.ones(len(X3_val)), np.zeros(len(X4_val))])

            th = youden_threshold(y_val, probs)
            if np.isfinite(th):
                thresholds.append(th)
        return float(np.median(thresholds)) if thresholds else 0.5

    opt_threshold = get_cv_threshold(model_name.lower())
 
    if model_name.lower() == "rf": 
        model = RandomForestClassifier(n_estimators=200, random_state=random_state, class_weight="balanced_subsample")
        model.fit(X_train_raw, y_train)
        y_score = model.predict_proba(X_test_raw)[:, 1]
        auc_val = roc_auc_score(y_test, y_score) if len(np.unique(y_test)) > 1 else np.nan
 
        return {"model": "rf", "mode": mode, "auc": float(auc_val), "threshold": opt_threshold}, \
               {"type": "sklearn_raw", "model": model, "feature_names": list(feature_names)}, (train_idx, test_idx), None

    elif model_name.lower() == "gam":
        model = LogisticGAM()
        model.fit(X_train_scaled, y_train)
        y_score = model.predict_proba(X_test_scaled)
        auc_val = roc_auc_score(y_test, y_score) if len(np.unique(y_test)) > 1 else np.nan
 
        return {"model": "gam", "mode": mode, "auc": float(auc_val), "threshold": opt_threshold}, \
               {"type": "pygam_scaled", "model": model, "feature_names": list(feature_names)}, (train_idx, test_idx), scaler

    elif model_name.lower() == "gaqq":
        SLDA_EBIC = ro.globalenv["SLDA.EBIC"]
        classify = ro.globalenv["classify"]

        X1, X2 = X_train_scaled[y_train == 1], X_train_scaled[y_train == 0]
        result = SLDA_EBIC(X1, X2, np.linspace(0.01, 5, 50), np.linspace(0.01, 5, 50))
        C_opt, delta_h_opt = np.array(result.rx2("C_opt")), np.array(result.rx2("delta_h_opt"))
        C_reg = C_opt + 1e-8 * np.eye(C_opt.shape[0])
        C_reg_r = ro.r.matrix(C_reg, nrow=C_reg.shape[0], ncol=C_reg.shape[1])

        X3, X4 = X_test_scaled[y_test == 1], X_test_scaled[y_test == 0]
        pred_result = classify(X1, X2, X3, X4, C_reg_r, delta_h_opt)

        if "y_hat" in pred_result.names:
            y_hat = np.array(pred_result.rx2("y_hat")).astype(float)
            y_score = (y_hat - y_hat.min()) / (y_hat.max() - y_hat.min() + 1e-12)
        else:
            y_score = 1.0 - np.array(pred_result.rx2("est_label")).astype(float)

        auc_val = roc_auc_score(y_test, y_score) if len(np.unique(y_test)) > 1 else np.nan

        gaqq_model = {
            "type": "gaqq", "classify": classify, "C_reg_r": C_reg_r, "delta_h_opt": delta_h_opt,
            "X1_train": X1, "X2_train": X2, "feature_names": list(feature_names)
        } 
        return {"model": "gaqq", "mode": mode, "auc": float(auc_val), "threshold": opt_threshold}, \
               gaqq_model, (train_idx, test_idx), scaler



def predict_and_evaluate_forecasting(
    model_obj, scaler, category_3_data, X_ref, label, model_name, 
    results_summary, fixed_threshold, save_dir="./results"
):
    os.makedirs(save_dir, exist_ok=True)
    
     
    category_mapping = {
        "1) Assoc w R": 1, "2) Assoc w R - Interim": 1,
        "4) Not assoc w R - Interim": 0, "5) Not assoc w R": 0
    }
    y_true = category_3_data["confidence_2023"].map(category_mapping).astype(int).values
    X_raw = category_3_data[X_ref.columns].fillna(0)  
     
    if model_name.lower() == "rf":
      
        y_probs = model_obj["model"].predict_proba(X_raw)[:, 1]
    else:
        
        X_scaled = scaler.transform(X_raw)
        
        if model_name.lower() == "gam":
            y_probs = model_obj["model"].predict_proba(X_scaled)
        
        elif model_name.lower() == "gaqq": 
            classify = model_obj["classify"]
            C_reg_r = model_obj["C_reg_r"]
            delta_h_opt = model_obj["delta_h_opt"] 
            X1, X2 = model_obj["X1_train"], model_obj["X2_train"] 
            X3 = X_scaled[y_true == 1]
            X4 = X_scaled[y_true == 0]
            
            pred_result = classify(X1, X2, X3, X4, C_reg_r, delta_h_opt)
            if "y_hat" in pred_result.names:
                y_hat = np.array(pred_result.rx2("y_hat")).astype(float)
                y_probs = (y_hat - y_hat.min()) / (y_hat.max() - y_hat.min() + 1e-12)
            else:
                est_label = np.array(pred_result.rx2("est_label")).astype(int)
                y_probs = 1.0 - est_label.astype(float)
 
    threshold = float(fixed_threshold)
    y_pred = (y_probs >= threshold).astype(int)
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    roc_auc = roc_auc_score(y_true, y_probs) if len(np.unique(y_true)) > 1 else float('nan')
    accuracy = np.mean(y_pred == y_true) * 100
    
    report_dict = classification_report(y_true, y_pred, labels=[0, 1], target_names=["Susceptible", "Resistant"], output_dict=True, zero_division=0)
     
    sens_low, sens_high = wilson_interval(tp, tp + fn) if (tp + fn) else (float('nan'), float('nan'))
    spec_low, spec_high = wilson_interval(tn, tn + fp) if (tn + fp) else (float('nan'), float('nan'))
 
    results_summary.append({
        "Feature Set": label,
        "Model": model_name,
        "Optimal Threshold": round(threshold, 3),
        "Accuracy (%)": accuracy,
        "ROC AUC": 100 * roc_auc if not math.isnan(roc_auc) else float('nan'),
        "Sensitivity (Res)": 100 * sensitivity,
        "Specificity (Sus)": 100 * specificity,
        "Precision (Res)": 100 * report_dict["Resistant"]["precision"],
        "Recall (Res)": 100 * report_dict["Resistant"]["recall"],
        "F1 (Res)": 100 * report_dict["Resistant"]["f1-score"],
        "Precision (Sus)": 100 * report_dict["Susceptible"]["precision"],
        "Recall (Sus)": 100 * report_dict["Susceptible"]["recall"],
        "F1 (Sus)": 100 * report_dict["Susceptible"]["f1-score"],
        "Support (Res)": int(report_dict["Resistant"]["support"]),
        "Support (Sus)": int(report_dict["Susceptible"]["support"]),
        "Incorrect Predictions": int((y_pred != y_true).sum()),
        "Sensitivity CI (95%)": (round(100*sens_low, 2), round(100*sens_high, 2)),
        "Specificity CI (95%)": (round(100*spec_low, 2), round(100*spec_high, 2)),
    })
    
    print(f"  [Success] AUC: {roc_auc:.3f} | Acc: {accuracy:.1f}% | Sens: {sensitivity:.2f} | Spec: {specificity:.2f}")
    return threshold
# def run_experiment_extended(X, y, model_name="rf", mode="holdout", label="Essential", test_size=0.2, random_state=42):
#     feature_names = X.columns
#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X)
#     y = np.asarray(y).astype(int)

#     train_idx, test_idx = train_test_split(
#         np.arange(len(X)), test_size=test_size, stratify=y, random_state=random_state
#     )
#     X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
#     y_train, y_test = y[train_idx], y[test_idx]

#     if model_name.lower() == "rf":
#         model = RandomForestClassifier(n_estimators=200, random_state=random_state, class_weight="balanced_subsample")
#         model.fit(X_train, y_train)
#         y_score = model.predict_proba(X_test)[:, 1]

#         thr = youden_threshold(y_test, y_score)
#         y_pred = (y_score >= thr).astype(int)

#         return {"model": "rf", "mode": mode, "auc": float(roc_auc_score(y_test, y_score)), "threshold": float(thr)}, \
#                {"type": "sklearn", "model": model, "feature_names": list(feature_names)}, (train_idx, test_idx), scaler

#     elif model_name.lower() == "gam":
#         model = LogisticGAM()
#         model.fit(X_train, y_train)
#         y_score = model.predict_proba(X_test)

#         thr = youden_threshold(y_test, y_score)
#         return {"model": "gam", "mode": mode, "auc": float(roc_auc_score(y_test, y_score)), "threshold": float(thr)}, \
#                {"type": "pygam", "model": model, "feature_names": list(feature_names)}, (train_idx, test_idx), scaler

#     elif model_name.lower() == "gaqq":
#         SLDA_EBIC = ro.globalenv["SLDA.EBIC"]
#         classify = ro.globalenv["classify"]

#         X1, X2 = X_train[y_train == 1], X_train[y_train == 0]
#         lambda1, lambda2 = np.linspace(0.01, 5, 50), np.linspace(0.01, 5, 50)

#         result = SLDA_EBIC(X1, X2, lambda1, lambda2)
#         C_opt, delta_h_opt = np.array(result.rx2("C_opt")), np.array(result.rx2("delta_h_opt"))

#         C_reg = C_opt + 1e-8 * np.eye(C_opt.shape[0])
#         C_reg_r = ro.r.matrix(C_reg, nrow=C_reg.shape[0], ncol=C_reg.shape[1])

#         X3, X4 = X_test[y_test == 1], X_test[y_test == 0]
#         pred_result = classify(X1, X2, X3, X4, C_reg_r, delta_h_opt)

#         if "y_hat" in pred_result.names:
#             y_hat = np.array(pred_result.rx2("y_hat")).astype(float)
#             y_score = (y_hat - y_hat.min()) / (y_hat.max() - y_hat.min() + 1e-12)
#         else:
#             est_label = np.array(pred_result.rx2("est_label")).astype(int)
#             y_score = 1.0 - est_label.astype(float)

#         thr = youden_threshold(y_test, y_score)
        
#         gaqq_model = {
#             "type": "gaqq", "SLDA_EBIC": SLDA_EBIC, "classify": classify,
#             "C_reg_r": C_reg_r, "delta_h_opt": delta_h_opt,
#             "X1_train": X1, "X2_train": X2, "feature_names": list(feature_names)
#         }
#         return {"model": "gaqq", "mode": mode, "auc": float(roc_auc_score(y_test, y_score)), "threshold": float(thr)}, \
#                gaqq_model, (train_idx, test_idx), scaler

# def predict_and_evaluate_forecasting(model_obj, category_3_data, X_ref, label, model_name, results_summary):
#     category_mapping = {"1) Assoc w R": 1, "2) Assoc w R - Interim": 1, "4) Not assoc w R - Interim": 0, "5) Not assoc w R": 0}
#     y_true = category_3_data["confidence_2023"].map(category_mapping).astype(int).values
#     X_cat3 = category_3_data[X_ref.columns].fillna(0).values

#     if model_obj["type"] in ("sklearn", "pygam"):
#         model = model_obj["model"]
#         y_score = model.predict_proba(X_cat3)[:, 1] if model_obj["type"] == "sklearn" else model.predict_proba(X_cat3)
#         thr = youden_threshold(y_true, y_score)
#         y_pred = (y_score >= thr).astype(int)
#         summarize_binary(y_true, y_pred, y_score, label, model_name, results_summary)
#         return thr

#     if model_obj["type"] == "gaqq":
#         classify, X1, X2 = model_obj["classify"], model_obj["X1_train"], model_obj["X2_train"]
#         C_reg_r, delta_h_opt = model_obj["C_reg_r"], model_obj["delta_h_opt"]
#         X3, X4 = X_cat3[y_true == 1], X_cat3[y_true == 0]

#         pred_result = classify(X1, X2, X3, X4, C_reg_r, delta_h_opt)
#         if "y_hat" in pred_result.names:
#             y_hat = np.array(pred_result.rx2("y_hat")).astype(float)
#             y_score = (y_hat - y_hat.min()) / (y_hat.max() - y_hat.min() + 1e-12)
#         else:
#             est_label = np.array(pred_result.rx2("est_label")).astype(int)
#             y_score = 1.0 - est_label.astype(float)

#         thr = youden_threshold(y_true, y_score)
#         y_pred = (y_score >= thr).astype(int)
#         summarize_binary(y_true, y_pred, y_score, label, model_name, results_summary)
#         return thr

In [13]:
# Load and align 2021/2023 datasets
catalog_2021 = pd.read_csv(file_2021).drop_duplicates()
catalog_2021 = catalog_2021[catalog_2021["Prox_3D"].notna()].copy()

df_2021 = pd.read_csv(file_2021)
df_2021 = df_2021[df_2021["Prox_3D"].notna()].copy()

df_2023 = pd.read_csv(file_2023)
df_2023 = df_2023[df_2023["Prox_3D"].notna()].copy()

# Generate label data and datasets
X_ess, y_ess, X_non, y_non, X_all, y_all, label_data, numeric_cols = create_label_data(catalog_2021)

datasets = {
    "Essential": (X_ess, y_ess, label_data[label_data["essentiality"] == "essential"].copy()),
    "Nonessential": (X_non, y_non, label_data[label_data["essentiality"] == "nonessential"].copy()),
    "Combined": (X_all, y_all, label_data.copy()),
}

# Build forecasting evaluation data
merged_df = merge_catalogs(df_2021, df_2023)
reclass_eval = reclassified_variants(merged_df, catalog_2021, X_ess)
cat3 = reclass_eval["category_3_data"].copy()

essential_genes = ["embB", "gyrA", "gyrB", "inhA", "katG", "rpoB", "rpsL", "tlyA"]
nonessential_genes = ["pncA", "gid", "ethA"]

cat3_map = {
    "Essential": cat3[cat3["gene"].isin(essential_genes)].copy(),
    "Nonessential": cat3[cat3["gene"].isin(nonessential_genes)].copy(),
    "Combined": cat3.copy(),
}

# Run experiments
all_results = []

for ds_name, ds_values in datasets.items(): 
    X, y = ds_values[0], ds_values[1]  
    
    print(f"\n{'='*15} DATASET: {ds_name} {'='*15}")
    for model_name in ["rf", "gam", "gaqq"]:
        print(f"--- Training {model_name.upper()} ---")
         
        res_train, model_obj, split_idx, scaler = run_experiment_extended(
            X, y, model_name=model_name, label=ds_name
        )
        opt_threshold = res_train["threshold"]
        print(f"  [Train] Derived Threshold: {opt_threshold:.3f}")
        
        # Forecasting  
        cat3_subset = cat3_map[ds_name]
        if cat3_subset.shape[0] > 0:
            print(f"--- Forecasting Evaluation: {model_name.upper()} ---")
             
            predict_and_evaluate_forecasting(
                model_obj=model_obj, 
                scaler=scaler,                     
                category_3_data=cat3_subset, 
                X_ref=X, 
                label=ds_name, 
                model_name=model_name.upper(), 
                results_summary=all_results, 
                fixed_threshold=opt_threshold      
            )
        else:
            print(f"  [Skip] No Category-3 samples for {ds_name}")


out_df = pd.DataFrame(all_results)
display(out_df)

 Genes with *only* uncertain confidence (excluded from training):
['ddn', 'rplC']

 Genes with labeled (certain) entries (used for training):
['embB', 'ethA', 'gid', 'gyrA', 'gyrB', 'inhA', 'katG', 'pncA', 'rpoB', 'rpsL', 'tlyA']

Filtered to non-uncertain entries: 382 rows

Repeated gene+mutation+label groups before collapse: 25
gene mutation_oneletter  binary_confidence  n_rows
gyrA              A384V                0.0       2
gyrA              A463S                0.0       2
gyrA               A90G                0.0       2
gyrA               A90V                1.0       2
gyrA               D94A                1.0       2
gyrA               D94G                1.0       2
gyrA               D94H                1.0       2
gyrA               D94N                1.0       2
gyrA               D94Y                1.0       2
gyrA               E21Q                0.0       2
Collapsed labeled supervised set: 370 -> 345 rows

Numeric feature count (after dropping metadata): 25

 No

,Feature Set,Model,Optimal Threshold,Accuracy (%),ROC AUC,Sensitivity (Res),Specificity (Sus),Precision (Res),Recall (Res),F1 (Res),Precision (Sus),Recall (Sus),F1 (Sus),Support (Res),Support (Sus),Incorrect Predictions,Sensitivity CI (95%),Specificity CI (95%)
0,Essential,RF,0.810,85.714286,100.000000,83.333333,100.0,100.000000,83.333333,90.909091,50.000000,100.0,66.666667,6,1,1,"(43.65, 96.99)","(20.65, 100.0)"
1,Essential,GAM,0.814,85.714286,100.000000,83.333333,100.0,100.000000,83.333333,90.909091,50.000000,100.0,66.666667,6,1,1,"(43.65, 96.99)","(20.65, 100.0)"
2,Essential,GAQQ,0.766,42.857143,100.000000,33.333333,100.0,100.000000,33.333333,50.000000,20.000000,100.0,33.333333,6,1,4,"(9.68, 70.0)","(20.65, 100.0)"
3,Nonessential,RF,0.935,63.636364,71.813725,62.745098,75.0,96.969697,62.745098,76.190476,13.636364,75.0,23.076923,51,4,20,"(49.02, 74.68)","(30.06, 95.44)"
4,Nonessential,GAM,0.920,36.363636,52.941176,33.333333,75.0,94.444444,33.333333,49.275362,8.108108,75.0,14.634146,51,4,35,"(21.97, 47.03)","(30.06, 95.44)"
5,Nonessential,GAQQ,0.446,70.909091,55.882353,76.470588,0.0,90.697674,76.470588,82.978723,0.000000,0.0,0.000000,51,4,16,"(63.24, 86.0)","(0.0, 48.99)"
6,Combined,RF,0.845,77.419355,73.508772,80.701754,40.0,93.877551,80.701754,86.792453,15.384615,40.0,22.222222,57,5,14,"(68.66, 88.87)","(11.76, 76.93)"
7,Combined,GAM,0.825,82.258065,55.789474,85.964912,40.0,94.230769,85.964912,89.908257,20.000000,40.0,26.666667,57,5,11,"(74.68, 92.71)","(11.76, 76.93)"
8,Combined,GAQQ,0.927,9.677419,50.526316,1.754386,100.0,100.000000,1.754386,3.448276,8.196721,100.0,15.151515,57,5,56,"(0.31, 9.29)","(56.55, 100.0)"


In [15]:
out_df = pd.DataFrame(all_results)
display(out_df)

,Feature Set,Model,Optimal Threshold,Accuracy (%),ROC AUC,Sensitivity (Res),Specificity (Sus),Precision (Res),Recall (Res),F1 (Res),Precision (Sus),Recall (Sus),F1 (Sus),Support (Res),Support (Sus),Incorrect Predictions,Sensitivity CI (95%),Specificity CI (95%)
0,Essential,RF,0.810,85.714286,100.000000,83.333333,100.0,100.000000,83.333333,90.909091,50.000000,100.0,66.666667,6,1,1,"(43.65, 96.99)","(20.65, 100.0)"
1,Essential,GAM,0.814,85.714286,100.000000,83.333333,100.0,100.000000,83.333333,90.909091,50.000000,100.0,66.666667,6,1,1,"(43.65, 96.99)","(20.65, 100.0)"
2,Essential,GAQQ,0.766,42.857143,100.000000,33.333333,100.0,100.000000,33.333333,50.000000,20.000000,100.0,33.333333,6,1,4,"(9.68, 70.0)","(20.65, 100.0)"
3,Nonessential,RF,0.935,63.636364,71.813725,62.745098,75.0,96.969697,62.745098,76.190476,13.636364,75.0,23.076923,51,4,20,"(49.02, 74.68)","(30.06, 95.44)"
4,Nonessential,GAM,0.920,36.363636,52.941176,33.333333,75.0,94.444444,33.333333,49.275362,8.108108,75.0,14.634146,51,4,35,"(21.97, 47.03)","(30.06, 95.44)"
5,Nonessential,GAQQ,0.446,70.909091,55.882353,76.470588,0.0,90.697674,76.470588,82.978723,0.000000,0.0,0.000000,51,4,16,"(63.24, 86.0)","(0.0, 48.99)"
6,Combined,RF,0.845,77.419355,73.508772,80.701754,40.0,93.877551,80.701754,86.792453,15.384615,40.0,22.222222,57,5,14,"(68.66, 88.87)","(11.76, 76.93)"
7,Combined,GAM,0.825,82.258065,55.789474,85.964912,40.0,94.230769,85.964912,89.908257,20.000000,40.0,26.666667,57,5,11,"(74.68, 92.71)","(11.76, 76.93)"
8,Combined,GAQQ,0.927,9.677419,50.526316,1.754386,100.0,100.000000,1.754386,3.448276,8.196721,100.0,15.151515,57,5,56,"(0.31, 9.29)","(56.55, 100.0)"


In [16]:
import os
os.makedirs("./results", exist_ok=True)  
out_df.to_csv("./results/comparison_model_mutation_level_summary.csv", index=False) 